# CSPM Finding Prioritization Model
## Azure 3-Tier Reference Workload — IIT Roorkee Capstone

**Source data:** `cspm_normalized_20260627.csv` — real Prowler v5.31.1 + ScoutSuite scan  
**Subscription:** `843e85de-3f5f-4614-967a-9f39f1fe9ba7` | **Tenant:** `e1a014d8-2ca2-4275-8220-414092cd8ed6`  
**Scan date:** 2026-06-27 | **Total findings:** 215 (128 FAIL / 87 PASS)

---

### Priority formula

$$\text{Priority} = \underbrace{\text{CVSS}_{\text{base}} \times w_1}_{\text{technical severity}} \times \underbrace{\text{Exposure} \times w_2}_{\text{attacker reach}} \times \underbrace{\text{BlastRadius} \times w_3}_{\text{post-exploit yield}} \quad \xrightarrow{\text{normalise}} \; [0, 100]$$

| Dimension | Source | Rationale |
|---|---|---|
| **CVSS base** | Prowler/ScoutSuite severity → NVD mapping | Technical exploitability, independent of environment |
| **Exposure** | Per-resource reachability score (0–1) | Same vuln ranks higher when the resource is internet-facing |
| **Blast radius** | Domain-level post-compromise yield (0–1) | IAM breach unlocks everything; Defender gap is policy-only |

> **Defender plan findings** are flagged as `scoped_out=True` — they require Microsoft Defender for Cloud Standard tier (paid) and are out of scope for a free-tier subscription. They appear in the model but are excluded from the action backlog.


## 1  Setup

In [ ]:
import re, json, warnings
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 70)

print("✓ Libraries loaded")


## 2  Load and parse real scan data

In [ ]:
DATA_FILE = Path("cspm_normalized_20260627.csv")
assert DATA_FILE.exists(), f"Put {DATA_FILE} in the same folder as this notebook"

df_raw = pd.read_csv(DATA_FILE)
print(f"✓ Loaded {len(df_raw)} findings from {DATA_FILE.name}")
print(f"  Columns : {list(df_raw.columns)}")
print(f"  Tools   : {df_raw['tool'].value_counts().to_dict()}")
print(f"  Status  : {df_raw['status'].value_counts().to_dict()}")
print(f"  Severity: {df_raw['severity'].value_counts().to_dict()}")

df_fail = df_raw[df_raw['status'].str.lower() == 'fail'].copy().reset_index(drop=True)
df_pass = df_raw[df_raw['status'].str.lower() == 'pass'].copy().reset_index(drop=True)
print(f"\n  FAIL rows : {len(df_fail)}")
print(f"  PASS rows : {len(df_pass)}")
print(f"  Unique checks : {df_raw['check_id'].nunique()}")
print(f"  Unique resources: {df_raw['resource_name'].nunique()}")


## 3  Dimension lookup tables

In [ ]:
# ── CVSS base score: NVD severity → numeric ───────────────────────────────────
CVSS_MAP = {"critical": 9.8, "high": 7.5, "medium": 5.5, "low": 3.0}

# ── Service → Security domain ─────────────────────────────────────────────────
SERVICE_DOMAIN = {
    "app": "Networking",         "App Services": "Networking",
    "appinsights": "Logging",
    "Azure Active Directory": "IAM",
    "defender": "Defender",      "Security Center": "Defender",
    "iam": "IAM",
    "Key Vault": "IAM",          "keyvault": "IAM",
    "Logging Monitoring": "Logging", "monitor": "Logging",
    "network": "Networking",
    "policy": "IAM",             "RBAC": "IAM",
    "SQL Database": "Database",  "sqlserver": "Database",
    "storage": "Storage",        "Storage Accounts": "Storage",
}

# ── Exposure: attacker reachability 0.0 (fully private) → 1.0 (internet) ─────
EXPOSURE_MAP = {
    "ref3tierst7mog1c":                   0.90,  # app storage, default-allow network rule
    "tfstatevignesh001":                  0.80,  # Terraform state backend, publicly reachable
    "ref3tierkv7mog1c":                   0.70,  # Key Vault management-plane accessible
    "ref3tier-dev-vnet/web-integration-subnet": 0.70,  # subnet without NSG
    "ref3tier-dev-web-nsg":               0.65,  # NSG resource
    "ref3tier-dev-sql-7mog1c":            0.65,  # SQL server (private endpoint)
    "SqlServers":                         0.65,
    "ref3tier-dev-db":                    0.60,
    "sql-admin-password":                 0.60,
    "ref3tier-dev-vnet":                  0.60,
    "ref3tier-dev-app-7mog1c":            0.55,  # App Service (HTTPS only)
    "843e85de-3f5f-4614-967a-9f39f1fe9ba7": 0.55,  # subscription-level
    "Containers":                         0.50,
    "NetworkWatcher_centralindia":        0.40,
    "authorizationPolicy":                0.40,
    "default":                            0.45,
    "security_contacts":                  0.30,
}
DEFENDER_EXPOSURE = 0.30   # Defender plan = subscription policy gap, not direct resource
SCOUTID_EXPOSURE  = 0.45   # ScoutSuite internal IDs — unknown resource type

def get_exposure(row):
    res = str(row["resource_name"])
    if res.startswith("Defender plan"): return DEFENDER_EXPOSURE
    if res.startswith("scoutid-"):      return SCOUTID_EXPOSURE
    return EXPOSURE_MAP.get(res, 0.50)

# ── Blast radius: post-compromise yield 0.0 → 1.0 ────────────────────────────
BLAST_MAP = {
    "IAM":       0.90,  # credential breach → access to all services
    "Storage":   0.80,  # data exfiltration, Terraform state exposure
    "Database":  0.75,  # data breach, PII exposure
    "Networking":0.75,  # lateral movement, traffic interception
    "Logging":   0.50,  # audit evasion, delayed detection
    "Defender":  0.40,  # threat detection gap — no direct exploitation path
    "Other":     0.35,
}

# ── Default weights (adjustable in sensitivity analysis) ──────────────────────
W1_CVSS, W2_EXPOSURE, W3_BLAST = 1.0, 1.0, 1.0

print("✓ Dimension tables defined")
print(f"  CVSS range    : {min(CVSS_MAP.values())} – {max(CVSS_MAP.values())}")
print(f"  Exposure range: {DEFENDER_EXPOSURE} – {max(EXPOSURE_MAP.values())}")
print(f"  Blast range   : {min(BLAST_MAP.values())} – {max(BLAST_MAP.values())}")


## 4  Compute priority scores

In [ ]:
def compute_scores(df: pd.DataFrame,
                   w1=W1_CVSS, w2=W2_EXPOSURE, w3=W3_BLAST) -> pd.DataFrame:
    out = df.copy()

    # ── Enrich with domain, compliance extracts ───────────────────────────────
    out["domain"]         = out["service"].map(SERVICE_DOMAIN).fillna("Other")
    out["cis_control"]    = out["compliance"].fillna("").apply(
        lambda s: "; ".join(re.findall(r'CIS-[\d.]+:\s*([\d.]+)', s)[:2]))
    out["mitre_technique"]= out["compliance"].fillna("").apply(
        lambda s: ", ".join(re.findall(r'MITRE-ATTACK:\s*(T\d+(?:\.\d+)?)', s)))

    # ── Three dimensions ─────────────────────────────────────────────────────
    out["cvss_base"]    = out["severity"].str.lower().map(CVSS_MAP).fillna(5.5)
    out["exposure"]     = out.apply(get_exposure, axis=1)
    out["blast_radius"] = out["domain"].map(BLAST_MAP).fillna(0.35)

    # ── Scope flag: Defender plan findings require paid Standard tier ─────────
    out["scoped_out"] = (
        out["service"].isin(["defender","Security Center"]) &
        out["resource_name"].str.startswith("Defender plan", na=False)
    )

    # ── Raw multiplicative score ──────────────────────────────────────────────
    out["raw_score"] = out["cvss_base"]*w1 * out["exposure"]*w2 * out["blast_radius"]*w3

    # ── Normalise to 0–100 ────────────────────────────────────────────────────
    max_raw = out["raw_score"].max()
    out["priority_score"] = (out["raw_score"] / max_raw * 100).round(1)

    # ── Priority tier ─────────────────────────────────────────────────────────
    out["priority_tier"] = pd.cut(
        out["priority_score"],
        bins=[0, 35, 60, 80, 100],
        labels=["P4 — Low","P3 — Medium","P2 — High","P1 — Critical"],
        right=True
    )

    return out.sort_values("priority_score", ascending=False).reset_index(drop=True)

df_scored = compute_scores(df_fail)

DISP = ["priority_score","priority_tier","scoped_out","severity","domain",
        "service","resource_name","check_id","cis_control","mitre_technique"]
print("✓ Scored 128 FAIL findings. Top 15:")
df_scored[DISP].head(15)


## 5  Tier summary and scoped-out analysis

In [ ]:
# ── Overall tier distribution ─────────────────────────────────────────────────
tier_df = (df_scored.groupby("priority_tier", observed=True)
    .agg(count=("priority_score","count"),
         avg_score=("priority_score","mean"),
         max_score=("priority_score","max"),
         scoped_out_count=("scoped_out","sum"))
    .round(1).reset_index())
tier_df.columns = ["Tier","Count","Avg Score","Max Score","Scoped Out"]
print("Priority tier distribution:")
print(tier_df.to_string(index=False))

# ── Scoped-out breakdown ──────────────────────────────────────────────────────
actionable = df_scored[~df_scored["scoped_out"]]
scoped     = df_scored[ df_scored["scoped_out"]]
print(f"\nActionable findings (in-scope): {len(actionable)}")
print(f"Scoped-out findings (Defender Standard required): {len(scoped)}")
print(f"\nTop 10 actionable findings:")
print(actionable[["priority_score","priority_tier","severity","domain",
                   "resource_name","check_id"]].head(10).to_string(index=False))


## 6  Domain breakdown

In [ ]:
domain_stats = (df_scored.groupby("domain")
    .agg(total=("check_id","count"),
         actionable=("scoped_out", lambda x: (~x).sum()),
         avg_score=("priority_score","mean"),
         max_score=("priority_score","max"),
         p1_count=("priority_tier", lambda x: (x=="P1 — Critical").sum()))
    .round(1).sort_values("avg_score", ascending=False).reset_index())
domain_stats.columns = ["Domain","Total","Actionable","Avg Score","Max Score","P1 Count"]
print("Domain breakdown:")
print(domain_stats.to_string(index=False))


## 7  Weight sensitivity analysis

Four scenarios modelling different organisational risk postures. Shows how the top-5 ranking shifts as weights change.


In [ ]:
SCENARIOS = {
    "Baseline (equal weights 1.0 / 1.0 / 1.0)":        (1.0, 1.0, 1.0),
    "Exposure-driven — perimeter focus (0.7/1.5/1.0)":  (0.7, 1.5, 1.0),
    "Blast-radius-driven — IAM/APT (1.0/1.0/1.5)":      (1.0, 1.0, 1.5),
    "CVSS-driven — compliance audit (1.5/0.8/0.8)":      (1.5, 0.8, 0.8),
}

print("=" * 95)
for scenario, (w1,w2,w3) in SCENARIOS.items():
    df_s = compute_scores(df_fail, w1=w1, w2=w2, w3=w3)
    actionable = df_s[~df_s["scoped_out"]]
    print(f"\n{scenario}")
    print(f"  weights: CVSS×{w1}  Exposure×{w2}  Blast×{w3}")
    for rank, (_, row) in enumerate(actionable.head(5).iterrows(), 1):
        print(f"  #{rank}  [{row['domain']:11}] score={row['priority_score']:5.1f}"
              f"  {row['resource_name']:<35}  {row['check_id'][:45]}")
print("\n" + "=" * 95)


## 8  Visualisations

In [ ]:
TIER_COLORS = {
    "P1 — Critical": "#d03b3b",
    "P2 — High":     "#c98500",
    "P3 — Medium":   "#2a78d6",
    "P4 — Low":      "#888780",
}
DOMAIN_COLORS = {
    "IAM":"#4a3aa7","Storage":"#2a78d6","Networking":"#1baf7a",
    "Database":"#eda100","Logging":"#888780","Defender":"#aaaaaa","Other":"#cccccc"
}
SCOPE_ALPHA = {True: 0.35, False: 1.0}

fig = plt.figure(figsize=(18, 13))
fig.suptitle("CSPM Prioritization — Azure 3-Tier Workload (128 FAILs, real scan data)",
             fontsize=14, fontweight="bold", y=0.98)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.42, wspace=0.35)

# ── Plot 1: Top 20 horizontal bar ────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, :2])
actionable_top = df_scored[~df_scored["scoped_out"]].head(20).iloc[::-1]
bar_colors = [TIER_COLORS.get(str(t),"#aaa") for t in actionable_top["priority_tier"]]
ax1.barh(range(len(actionable_top)), actionable_top["priority_score"],
         color=bar_colors, height=0.65)
labels = [f"{row['check_id'][:30]}  [{row['resource_name'][:22]}]"
          for _, row in actionable_top.iterrows()]
ax1.set_yticks(range(len(actionable_top)))
ax1.set_yticklabels(labels, fontsize=8)
for i,(sc,tier) in enumerate(zip(actionable_top["priority_score"],
                                  actionable_top["priority_tier"])):
    ax1.text(sc+0.8, i, f"{sc:.0f}", va="center", fontsize=8, fontweight="bold",
             color=TIER_COLORS.get(str(tier),"#aaa"))
ax1.set_xlim(0,120); ax1.set_xlabel("Priority score (0–100)")
ax1.set_title("Top 20 actionable findings (scoped-out Defender excluded)", fontsize=11, pad=8)
ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)
patches = [mpatches.Patch(color=c,label=l) for l,c in TIER_COLORS.items()]
ax1.legend(handles=patches, loc="lower right", fontsize=8)

# ── Plot 2: Tier donut ────────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 2])
tier_vc = df_scored["priority_tier"].value_counts().sort_index(ascending=False)
tc = [TIER_COLORS.get(str(t),"#aaa") for t in tier_vc.index]
wedges,texts,autotexts = ax2.pie(tier_vc.values, labels=tier_vc.index, colors=tc,
    autopct="%d%%", startangle=90, pctdistance=0.75,
    wedgeprops={"width":0.52,"edgecolor":"white","linewidth":2})
for t in texts: t.set_fontsize(8)
for at in autotexts:
    at.set_fontsize(8.5); at.set_fontweight("bold"); at.set_color("white")
ax2.set_title("All 128 FAILs by tier", fontsize=11, pad=8)

# ── Plot 3: Bubble — CVSS × Exposure (bubble = blast radius) ─────────────────
ax3 = fig.add_subplot(gs[1, :2])
for domain, grp in df_scored.groupby("domain"):
    sizes  = (grp["blast_radius"] * 400).clip(lower=30)
    alphas = [SCOPE_ALPHA[s] for s in grp["scoped_out"]]
    for _, row in grp.iterrows():
        ax3.scatter(row["cvss_base"], row["exposure"],
                    s=row["blast_radius"]*400,
                    c=DOMAIN_COLORS.get(domain,"#aaa"),
                    alpha=SCOPE_ALPHA[row["scoped_out"]],
                    edgecolors="white", linewidths=0.8)
for _, row in df_scored[(df_scored["priority_score"]>=75) & ~df_scored["scoped_out"]].iterrows():
    ax3.annotate(row["resource_name"][:18], (row["cvss_base"],row["exposure"]),
                 fontsize=7, ha="center", va="bottom",
                 xytext=(0,5), textcoords="offset points", color="#333")
ax3.axvline(7.0,color="#ddd",linestyle="--",linewidth=0.8,zorder=0)
ax3.axhline(0.65,color="#ddd",linestyle="--",linewidth=0.8,zorder=0)
ax3.set_xlabel("CVSS base score", fontsize=10)
ax3.set_ylabel("Exposure score", fontsize=10)
ax3.set_xlim(0,11.5); ax3.set_ylim(0,1.1)
ax3.set_title("CVSS × Exposure — bubble = blast radius  (faded = scoped-out)", fontsize=11, pad=8)
legend_els = [mpatches.Patch(color=c,label=d) for d,c in DOMAIN_COLORS.items()]
ax3.legend(handles=legend_els, loc="lower right", fontsize=8, ncol=2)
ax3.spines["top"].set_visible(False); ax3.spines["right"].set_visible(False)

# ── Plot 4: Domain avg score ──────────────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 2])
domain_avg = df_scored.groupby("domain")["priority_score"].mean().sort_values()
dc = [DOMAIN_COLORS.get(d,"#aaa") for d in domain_avg.index]
ax4.barh(domain_avg.index, domain_avg.values, color=dc, height=0.6)
for i,(d,v) in enumerate(domain_avg.items()):
    ax4.text(v+0.5,i,f"{v:.1f}",va="center",fontsize=8.5,fontweight="bold")
ax4.set_xlabel("Avg priority score")
ax4.set_title("Average score by domain", fontsize=11, pad=8)
ax4.set_xlim(0,105)
ax4.spines["top"].set_visible(False); ax4.spines["right"].set_visible(False)

plt.savefig("prioritization_model.png", dpi=150, bbox_inches="tight",
            facecolor="white", edgecolor="none")
plt.show()
print("✓ Figure saved → prioritization_model.png")


## 9  Weight sensitivity heatmap

How does the top-scoring actionable finding change as we sweep w₁ (CVSS) and w₂ (Exposure)?

In [ ]:
# Use the highest-priority actionable finding
top_finding = df_scored[~df_scored["scoped_out"]].iloc[0]
print(f"Heatmap target: [{top_finding['resource_name']}] {top_finding['check_id']}")

w1_vals = np.arange(0.5, 2.1, 0.25)
w2_vals = np.arange(0.5, 2.1, 0.25)
heat    = np.zeros((len(w2_vals), len(w1_vals)))

for i, w2 in enumerate(w2_vals):
    for j, w1 in enumerate(w1_vals):
        tmp = compute_scores(df_fail, w1=w1, w2=w2, w3=1.0)
        row = tmp[tmp["resource_name"] == top_finding["resource_name"]]
        heat[i, j] = row["priority_score"].values[0] if len(row) else 0

cmap = LinearSegmentedColormap.from_list("risk",["#2a78d6","#eda100","#d03b3b"])
fig, ax = plt.subplots(figsize=(8,5))
im = ax.imshow(heat, cmap=cmap, aspect="auto", vmin=0, vmax=100,
               extent=[w1_vals[0]-0.125, w1_vals[-1]+0.125,
                       w2_vals[0]-0.125, w2_vals[-1]+0.125],
               origin="lower")
plt.colorbar(im, ax=ax, label="Priority score (0–100)")
ax.set_xlabel("CVSS weight (w₁)", fontsize=10)
ax.set_ylabel("Exposure weight (w₂)", fontsize=10)
ax.set_title(f"Score sensitivity: {top_finding['resource_name']}\n(w₃ blast = 1.0 fixed)", fontsize=11)
ax.axvline(1.0, color="white", linewidth=1.5, linestyle="--", alpha=0.7)
ax.axhline(1.0, color="white", linewidth=1.5, linestyle="--", alpha=0.7)
ax.text(1.05, 0.55, "baseline", color="white", fontsize=8.5)
plt.tight_layout()
plt.savefig("sensitivity_heatmap.png", dpi=150, bbox_inches="tight", facecolor="white")
plt.show()
print("✓ Heatmap saved → sensitivity_heatmap.png")


## 10  Export scored findings

In [ ]:
EXPORT_COLS = [
    "tool","scan_date","check_id","check_title","status","severity",
    "service","resource_name","region","domain","scoped_out",
    "cvss_base","exposure","blast_radius","raw_score","priority_score","priority_tier",
    "cis_control","mitre_technique","description","remediation","compliance",
    "subscription_id"
]
# Add pass rows (unscored) for completeness
df_pass_out = df_pass.copy()
for col in ["domain","cis_control","mitre_technique","cvss_base",
            "exposure","blast_radius","raw_score","priority_score","scoped_out"]:
    df_pass_out[col] = None
df_pass_out["priority_tier"] = "PASS"

export_cols_available = [c for c in EXPORT_COLS if c in df_scored.columns]
df_export = pd.concat([
    df_scored[export_cols_available],
    df_pass_out[[c for c in export_cols_available if c in df_pass_out.columns]]
], ignore_index=True)

out_path = "cspm_scored_findings.csv"
df_export.to_csv(out_path, index=False)
print(f"✓ Exported {len(df_export)} findings → {out_path}")

print("\nP1 — Critical actionable findings:")
p1 = df_scored[(df_scored["priority_tier"].astype(str)=="P1 — Critical") &
               ~df_scored["scoped_out"]]
for _, r in p1.iterrows():
    print(f"  score={r['priority_score']:5.1f}  [{r['domain']:10}]"
          f"  {r['resource_name']:<35}  {r['check_id'][:50]}")


## 11  Remediation roadmap

In [ ]:
EFFORT_MAP = {"critical":4,"high":8,"medium":16,"low":24}

roadmap = df_scored[~df_scored["scoped_out"]].copy()
roadmap["sprint"] = pd.cut(
    roadmap["priority_score"],
    bins=[0,35,60,80,100],
    labels=["Sprint 3 (backlog)","Sprint 2 (planned)","Sprint 1 (next)","Sprint 0 (immediate)"],
    right=True
)
roadmap["est_hours"] = roadmap["severity"].str.lower().map(EFFORT_MAP).fillna(16)

print("=" * 100)
for sprint in ["Sprint 0 (immediate)","Sprint 1 (next)","Sprint 2 (planned)","Sprint 3 (backlog)"]:
    items = roadmap[roadmap["sprint"].astype(str)==sprint]
    if items.empty: continue
    print(f"\n{sprint}  ({len(items)} findings, ~{int(items['est_hours'].sum())}h)")
    print("-" * 100)
    for _, r in items.iterrows():
        print(f"  score={r['priority_score']:5.1f}  [{r['domain']:10}]  "
              f"{r['resource_name']:<32}  {r['check_id'][:48]}")

total_h = int(roadmap["est_hours"].sum())
print(f"\n{'='*100}")
print(f"Total actionable effort: ~{total_h}h across {len(roadmap)} findings")
print(f"Scoped-out (Defender Standard):  {df_scored['scoped_out'].sum()} findings — require paid tier")


## 12  LLM-assisted triage via Anthropic API

Enrich P1/P2 findings with Claude-generated risk narrative, attack chain, effort estimate, and Terraform fix.  
Set the `ANTHROPIC_API_KEY` environment variable before running.


In [ ]:
import os, time

try:
    import anthropic
    ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY","")
    if ANTHROPIC_API_KEY:
        client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
        print("✓ Anthropic client ready")
    else:
        print("⚠  ANTHROPIC_API_KEY not set — live triage disabled")
        print("   PowerShell: $env:ANTHROPIC_API_KEY = 'sk-ant-...'")
except ImportError:
    print("⚠  pip install anthropic  then re-run")
    ANTHROPIC_API_KEY = ""

WORKLOAD_CTX = """
WORKLOAD: Azure 3-Tier (App Service web tier → Azure SQL private endpoint → Blob Storage + Key Vault)
IaC: Terraform azurerm ~> 3.110 | Subscription: 843e85de-3f5f-4614-967a-9f39f1fe9ba7
Compliance target: CIS Azure Benchmark v5.0 | Tools: Prowler v5.31.1 + ScoutSuite
""".strip()

def build_triage_prompt(row: pd.Series) -> str:
    return f"""You are a senior Azure cloud security architect (10 years experience).
Triage this CSPM finding and respond ONLY with valid JSON matching the schema below.

FINDING
-------
Check ID      : {row['check_id']}
Check title   : {row.get('check_title','—')}
Resource      : {row['resource_name']}
Service       : {row['service']}
Domain        : {row['domain']}
Severity      : {row['severity'].upper()}
CVSS base     : {row['cvss_base']}
Exposure      : {row['exposure']} (0=private 1=internet-facing)
Blast radius  : {row['blast_radius']} (0=isolated 1=full-workload)
Priority score: {row['priority_score']}/100
Priority tier : {row['priority_tier']}
CIS control   : {row.get('cis_control','—')}
MITRE ATT&CK  : {row.get('mitre_technique','—')}
Description   : {str(row.get('description','—'))[:200]}
Remediation   : {str(row.get('remediation','—'))[:200]}

{WORKLOAD_CTX}

JSON SCHEMA (no markdown, no preamble — raw JSON only):
{{
  "risk_narrative": "2–3 sentence plain-English risk description for a technical manager",
  "attack_chain":   "Numbered steps an attacker takes from initial access to impact",
  "effort":         "Low|Medium|High",
  "effort_detail":  "One sentence on fix complexity and dependencies",
  "terraform_fix":  "Exact minimal Terraform HCL to remediate",
  "verify_cmd":     "Azure CLI command to verify the fix post-apply",
  "false_positive": "One sentence on how to confirm this is a genuine finding"
}}"""

def triage(row: pd.Series) -> dict:
    if not ANTHROPIC_API_KEY:
        return {"error": "ANTHROPIC_API_KEY not set"}
    try:
        msg = client.messages.create(
            model="claude-sonnet-4-6", max_tokens=700,
            messages=[{"role":"user","content":build_triage_prompt(row)}]
        )
        return json.loads(msg.content[0].text.strip())
    except Exception as e:
        return {"error": str(e)}

# ── Triage top 3 actionable P1 findings ──────────────────────────────────────
p1_actionable = df_scored[(df_scored["priority_tier"].astype(str)=="P1 — Critical") &
                           ~df_scored["scoped_out"]].head(3)

if ANTHROPIC_API_KEY:
    print(f"Triaging {len(p1_actionable)} P1 findings...\n")
    for _, row in p1_actionable.iterrows():
        print(f"► {row['resource_name']} / {row['check_id']} (score={row['priority_score']})")
        result = triage(row)
        if "error" not in result:
            print(f"  Risk   : {result.get('risk_narrative','')[:110]}...")
            print(f"  Effort : {result.get('effort','—')} — {result.get('effort_detail','')[:80]}")
            print(f"  TF fix : {result.get('terraform_fix','')[:90]}...")
        else:
            print(f"  Error  : {result['error']}")
        print(); time.sleep(1)
else:
    # Show sample prompt so cell is useful without the key
    sample = df_scored[~df_scored["scoped_out"]].iloc[0]
    print("Sample prompt that would be sent for the top finding:")
    print("-"*70)
    print(build_triage_prompt(sample)[:800]+"\n...")


## 13  What is already secure (PASS analysis)

In [ ]:
print("Controls PASSING — evidence of Terraform hardening working:\n")
key_passes = [
    "sqlserver_unrestricted_inbound_access",
    "app_minimum_tls_version_12",
    "keyvault_rbac_enabled",
    "storage_blob_public_access_level_is_disabled",
    "sqlserver_tde_encryption_enabled",
    "iam_role_user_access_admin_restricted",
    "network_rdp_internet_access_restricted",
    "network_ssh_internet_access_restricted",
    "app_ftp_deployment_disabled",
    "app_ensure_http_is_redirected_to_https",
    "app_register_with_identity",
    "sqlserver_recommended_minimal_tls_version",
    "storage_cross_tenant_replication_disabled",
    "storage_blob_versioning_is_enabled",
    "network_subnet_nsg_associated",
]
pass_found = df_pass[df_pass["check_id"].isin(key_passes)]
for _, r in pass_found[["check_id","severity","service"]].drop_duplicates("check_id").iterrows():
    print(f"  ✓ [PASS] [{r['severity']:8}] {r['check_id']}")

print(f"\n  {len(pass_found)} hardening controls confirmed active out of 87 total PASSes")
